# Cross-Thread Coordination Features

**Goal:** Build a user-user temporal co-occurrence graph from Karin's Pantip dataset, extract per-user graph features, and integrate them with the existing Karin/Lindsey feature stack across the three group-split experiments.

**Input files** (Karin's preprocessed outputs, all UTF-8 with BOM):
- `00_Datasets.txt` — list of post records (see Cell 2 for schema)
- `01_SockpuppetGroup.txt` — `List[List[user_id]]`, sockpuppet IP groups
- `02_NormalGroup.txt` — `List[List[user_id]]`, normal user groups (paired controls)
- `0{3,4,5}_*_{Train,Test}_{Sock,Normal}.txt` — same nested-list format; experiments 1/2/3 split groups (not individual users) into train/test

**Group-level split detail (sanity-checked from the upload):**

| Exp | Train groups | Test groups | Train users | Test users |
|---|---|---|---|---|
| 1 (20/80) | 6 sock + 6 norm | 25 sock + 25 norm | 43+43 | 97+97 |
| 2 (50/50) | 11 sock + 11 norm | 20 sock + 20 norm | 70+70 | 70+70 |
| 3 (65/35) | 20 sock + 20 norm | 11 sock + 11 norm | 99+99 | 41+41 |

Train ∪ Test = master, no group leaks across splits.

**Output:** `coordination_features_exp{1,2,3}.csv` — one row per graph user, ready to be joined with existing Karin+Lindsey feature matrices on `user_id`.

## Cell 1 — Imports and config

In [ ]:
import json
from pathlib import Path
from collections import defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import networkx as nx

# Paths — adjust to your project layout
DATA_DIR  = Path('../data')
POSTS_FILE = DATA_DIR / '00_Datasets.txt'

SOCK_MASTER = DATA_DIR / '01_SockpuppetGroup.txt'
NORM_MASTER = DATA_DIR / '02_NormalGroup.txt'

SPLIT_FILES = {
    1: {
        'train_sock':   DATA_DIR / '03_First_Train_Sock.txt',
        'test_sock':    DATA_DIR / '03_First_Test_Sock.txt',
        'train_normal': DATA_DIR / '03_First_Train_Normal.txt',
        'test_normal':  DATA_DIR / '03_First_Test_Normal.txt',
    },
    2: {
        'train_sock':   DATA_DIR / '04_Second_Train_Sock.txt',
        'test_sock':    DATA_DIR / '04_Second_Test_Sock.txt',
        'train_normal': DATA_DIR / '04_Second_Train_Normal.txt',
        'test_normal':  DATA_DIR / '04_Second_Test_Normal.txt',
    },
    3: {
        'train_sock':   DATA_DIR / '05_Third_Train_Sock.txt',
        'test_sock':    DATA_DIR / '05_Third_Test_Sock.txt',
        'train_normal': DATA_DIR / '05_Third_Train_Normal.txt',
        'test_normal':  DATA_DIR / '05_Third_Test_Normal.txt',
    },
}

OUT_DIR = Path('./out_coordination'); OUT_DIR.mkdir(exist_ok=True)

# Hyperparameters (tune in sensitivity sweep)
TIME_WINDOWS_MIN     = [30, 120, 360]
MIN_SHARED_THREADS   = 2
MAX_USERS_PER_THREAD = 300
RANDOM_SEED          = 42

# Karin's files ship with a UTF-8 BOM
def load_json(path):
    with open(path, 'r', encoding='utf-8-sig') as f:
        return json.load(f)

np.random.seed(RANDOM_SEED)
print('Config loaded')

## Cell 2 — Load Karin's post-level data

Per-post schema (from `00_Datasets.txt`):

| Field | Type | Notes |
|---|---|---|
| `num_0` | int | position in thread (0 = opener) |
| `num_1`, `num_r` | int/str | sub-reply ordering |
| `thread_id` | int | Pantip topic id |
| `thread_name` | str | |
| `user_id`, `user_name` | int / str | |
| `messages` | str | post text |
| `length` | int | character count |
| `helpful` | int/str | reaction count |
| `tag` | list[str] | e.g. `['การเมือง']` |
| `datetime` | str | `'%m/%d/%Y %H:%M:%S'` |
| `user_sameIP` | list[int] | other user_ids sharing this user's IP |
| `IP` | list[str] | raw IPs |
| `IP_check` | str/bool | `'Yes'` if this post is from a known IP-shared group |
| `user_type` | str/NaN | annotation tag (`'sms'`, etc.) |
| `profile_like`, `emo_like` | list | engagement metadata |

In [ ]:
records = load_json(POSTS_FILE)
posts = pd.DataFrame(records)

posts['user_id']   = posts['user_id'].astype(int)
posts['thread_id'] = posts['thread_id'].astype(int)
posts['datetime']  = pd.to_datetime(posts['datetime'], format='%m/%d/%Y %H:%M:%S')
posts = posts.sort_values(['thread_id', 'datetime']).reset_index(drop=True)

print(f'Posts:   {len(posts):,}')
print(f'Users:   {posts.user_id.nunique():,}')
print(f'Threads: {posts.thread_id.nunique():,}')
print(f'Date range: {posts.datetime.min()} → {posts.datetime.max()}')

## Cell 3 — Load labels and group-level splits

Labels come from the master group files (`01_SockpuppetGroup`, `02_NormalGroup`). Splits are at the **group** level: a user inherits the train/test tag of their group. We flatten each split into per-user sets.

In [ ]:
sock_groups_master = load_json(SOCK_MASTER)
norm_groups_master = load_json(NORM_MASTER)

sock_users_master = {u for g in sock_groups_master for u in g}
norm_users_master = {u for g in norm_groups_master for u in g}

label_map = {u: 1 for u in sock_users_master}
label_map.update({u: 0 for u in norm_users_master})

print(f'Sockpuppet groups (master): {len(sock_groups_master)}, users: {len(sock_users_master)}')
print(f'Normal groups (master):     {len(norm_groups_master)}, users: {len(norm_users_master)}')
print(f'Total labeled users:        {len(label_map)}')

# user_id → IP-group tag (used in Cell 10)
user_to_group = {}
for gid, members in enumerate(sock_groups_master):
    for u in members:
        user_to_group[u] = f'sock_{gid}'
for gid, members in enumerate(norm_groups_master):
    for u in members:
        user_to_group[u] = f'norm_{gid}'

In [ ]:
def load_split(exp_id: int):
    sp = SPLIT_FILES[exp_id]
    train_sock   = {u for g in load_json(sp['train_sock'])   for u in g}
    test_sock    = {u for g in load_json(sp['test_sock'])    for u in g}
    train_normal = {u for g in load_json(sp['train_normal']) for u in g}
    test_normal  = {u for g in load_json(sp['test_normal'])  for u in g}

    train_users = train_sock | train_normal
    test_users  = test_sock  | test_normal

    overlap = train_users & test_users
    assert not overlap, f'Exp {exp_id} has {len(overlap)} user(s) in both train and test'

    return {
        'train_users': train_users,  'test_users':  test_users,
        'train_sock':  train_sock,   'test_sock':   test_sock,
        'train_normal': train_normal,'test_normal':  test_normal,
    }

splits = {exp: load_split(exp) for exp in (1, 2, 3)}
for exp, s in splits.items():
    print(f'Exp {exp}: train={len(s["train_users"]):>3} '
          f'(sock={len(s["train_sock"])} norm={len(s["train_normal"])})  '
          f'test={len(s["test_users"]):>3} '
          f'(sock={len(s["test_sock"])} norm={len(s["test_normal"])})')

## Cell 4 — Build user → activity and thread → activity indexes

Graph includes ALL users in the posts file, not just labeled ones. Unlabeled users can be neighbors of labeled users and contribute to coordination signal.

In [ ]:
user_activity   = defaultdict(list)
thread_activity = defaultdict(list)

for row in posts.itertuples(index=False):
    user_activity[row.user_id].append((row.thread_id, row.datetime))
    thread_activity[row.thread_id].append((row.user_id, row.datetime))

for uid in user_activity:
    user_activity[uid].sort(key=lambda x: x[1])
for tid in thread_activity:
    thread_activity[tid].sort(key=lambda x: x[1])

print(f'Users in index:   {len(user_activity):,}')
print(f'Threads in index: {len(thread_activity):,}')

ppu = pd.Series([len(v) for v in user_activity.values()])
ppt = pd.Series([len(v) for v in thread_activity.values()])
print(f'\nPosts/user:   median={ppu.median():.0f}  p90={ppu.quantile(0.9):.0f}  max={ppu.max()}')
print(f'Posts/thread: median={ppt.median():.0f}  p90={ppt.quantile(0.9):.0f}  max={ppt.max()}')

labeled_in_posts = set(user_activity) & set(label_map)
print(f'\nLabeled users with ≥1 post: {len(labeled_in_posts)} / {len(label_map)}')
missing = set(label_map) - set(user_activity)
if missing:
    print(f'⚠️  {len(missing)} labeled users have no posts: {sorted(missing)[:10]}'
          f'{"..." if len(missing) > 10 else ""}')

## Cell 5 — Build user-user co-occurrence statistics

Per thread, enumerate user pairs. For each pair record: shared-thread count + list of min time gaps (using each user's first post in the thread as the timestamp). Threads with > `MAX_USERS_PER_THREAD` posters are sub-sampled to the earliest N.

In [ ]:
pair_stats = {}
n_truncated = 0

for tid, events in thread_activity.items():
    user_first = {}
    for uid, ts in events:
        if uid not in user_first:
            user_first[uid] = ts

    if len(user_first) > MAX_USERS_PER_THREAD:
        items = sorted(user_first.items(), key=lambda kv: kv[1])[:MAX_USERS_PER_THREAD]
        user_first = dict(items)
        n_truncated += 1

    users = sorted(user_first)
    for u1, u2 in combinations(users, 2):
        gap = abs((user_first[u1] - user_first[u2]).total_seconds())
        key = (u1, u2)
        s = pair_stats.get(key)
        if s is None:
            pair_stats[key] = {'n_threads': 1, 'min_gaps_sec': [gap]}
        else:
            s['n_threads'] += 1
            s['min_gaps_sec'].append(gap)

print(f'Raw pairs (≥1 shared thread): {len(pair_stats):,}')
print(f'Threads truncated to {MAX_USERS_PER_THREAD}: {n_truncated}')

pair_stats = {k: v for k, v in pair_stats.items() if v['n_threads'] >= MIN_SHARED_THREADS}
print(f'After MIN_SHARED_THREADS={MIN_SHARED_THREADS}: {len(pair_stats):,}')

## Cell 6 — Sparse adjacency (full + per time-window)

In [ ]:
graph_users = sorted({u for pair in pair_stats for u in pair})
u2i = {u: i for i, u in enumerate(graph_users)}
N = len(graph_users)
print(f'Graph users: {N:,}')

def build_adj(window_seconds=None):
    rows, cols, vals = [], [], []
    for (u1, u2), s in pair_stats.items():
        if window_seconds is None:
            w = s['n_threads']
        else:
            w = sum(1 for g in s['min_gaps_sec'] if g <= window_seconds)
        if w < MIN_SHARED_THREADS:
            continue
        i, j = u2i[u1], u2i[u2]
        rows += [i, j]; cols += [j, i]; vals += [w, w]
    return csr_matrix((vals, (rows, cols)), shape=(N, N), dtype=np.float32)

adj_all     = build_adj(None)
adj_windows = {w: build_adj(w * 60) for w in TIME_WINDOWS_MIN}

print(f'Edges (all):       {adj_all.nnz // 2:,}')
for w, A in adj_windows.items():
    print(f'Edges ({w:>3}-min):    {A.nnz // 2:,}')

## Cell 7 — Label-free graph features (computed once, reused across experiments)

In [ ]:
def degrees(A):
    return np.asarray(A.sum(axis=1)).ravel()

feat = pd.DataFrame({'user_id': graph_users})

feat['deg_all'] = degrees(adj_all)
for w, A in adj_windows.items():
    feat[f'deg_w{w}'] = degrees(A)

edge_weights = np.array(adj_all[adj_all.nonzero()]).ravel()
p90 = float(np.percentile(edge_weights, 90)) if len(edge_weights) else 0.0
print(f'p90 edge weight: {p90:.2f}')
strong = (adj_all >= p90).astype(np.int8)
feat['n_strong_neighbors'] = np.asarray(strong.sum(axis=1)).ravel()

feat['top_neighbor_weight'] = np.asarray(adj_all.max(axis=1).todense()).ravel()

G_bin = nx.from_scipy_sparse_array((adj_all > 0).astype(np.int8))
clust = nx.clustering(G_bin)
feat['clustering_coef'] = feat.index.map(lambda i: clust.get(i, 0.0)).astype(float)

deg_vec = feat['deg_all'].values
A_lil = adj_all.tolil()
mean_nb = np.zeros(N)
for i in range(N):
    nbrs = A_lil.rows[i]
    if nbrs:
        mean_nb[i] = deg_vec[nbrs].mean()
feat['mean_neighbor_degree'] = mean_nb

tightest = min(TIME_WINDOWS_MIN)
feat['temporal_concentration'] = (
    feat[f'deg_w{tightest}'] / feat['deg_all'].replace(0, np.nan)
).fillna(0)

print(feat.describe())

## Cell 8 — Label-dependent features per experiment (leakage-safe)

`neighbor_sockpuppet_ratio` and friends use **only training-set labels**. Test users are evaluated using their neighbors' training labels (which is allowed — neighbors aren't the test user).

In [ ]:
def compute_label_dependent(feat_base, exp_split, label_map_in):
    train_users = exp_split['train_users']

    train_positive = np.zeros(N, dtype=np.float32)
    train_known    = np.zeros(N, dtype=np.float32)
    for i, uid in enumerate(graph_users):
        if uid in train_users:
            train_known[i] = 1.0
            if label_map_in.get(uid, 0) == 1:
                train_positive[i] = 1.0

    neighbor_pos   = adj_all.dot(train_positive)
    neighbor_known = adj_all.dot(train_known)

    ratio = np.divide(
        neighbor_pos, neighbor_known,
        out=np.zeros_like(neighbor_pos),
        where=neighbor_known > 0,
    )
    out = feat_base.copy()
    out['neighbor_sockpuppet_ratio']  = ratio
    out['n_known_train_neighbors']    = neighbor_known
    out['n_positive_train_neighbors'] = neighbor_pos
    return out

for exp in (1, 2, 3):
    s = splits[exp]
    feat_exp = compute_label_dependent(feat, s, label_map)
    def tag(u, s=s):
        if u in s['train_users']: return 'train'
        if u in s['test_users']:  return 'test'
        return 'unused'
    feat_exp['split'] = feat_exp.user_id.map(tag)
    feat_exp['label'] = feat_exp.user_id.map(lambda u: label_map.get(u, -1))
    out_path = OUT_DIR / f'coordination_features_exp{exp}.csv'
    feat_exp.to_csv(out_path, index=False)
    n_tr = (feat_exp.split == 'train').sum()
    n_te = (feat_exp.split == 'test').sum()
    print(f'Exp {exp}: wrote {out_path}  ({len(feat_exp):,} users; train={n_tr}, test={n_te})')

## Cell 9 — Leakage sanity check (REQUIRED before trusting results)

Shuffle labels among the labeled pool, recompute `neighbor_sockpuppet_ratio`, and confirm its AUC drops to ~0.5. If it stays high, there's a leak.

In [ ]:
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(RANDOM_SEED)
exp_id = 1
s = splits[exp_id]

feat_real = compute_label_dependent(feat, s, label_map)

all_lbl_users = list(s['train_users'] | s['test_users'])
true_vec = np.array([label_map[u] for u in all_lbl_users])
shuf_vec = rng.permutation(true_vec)
shuf_map = {u: int(l) for u, l in zip(all_lbl_users, shuf_vec)}
feat_shuf = compute_label_dependent(feat, s, {**label_map, **shuf_map})

def auc_on_test(df, lbl_map):
    test_df = df[df.user_id.isin(s['test_users'])].copy()
    test_df['y'] = test_df.user_id.map(lbl_map)
    if test_df.y.nunique() < 2 or test_df.neighbor_sockpuppet_ratio.nunique() < 2:
        return float('nan')
    return roc_auc_score(test_df.y, test_df.neighbor_sockpuppet_ratio)

auc_real = auc_on_test(feat_real, label_map)
auc_shuf = auc_on_test(feat_shuf, {**label_map, **shuf_map})
print(f'neighbor_sockpuppet_ratio AUC on test, Exp {exp_id}:')
print(f'  Real labels:     {auc_real:.3f}   (should be > 0.6 to be useful)')
print(f'  Shuffled labels: {auc_shuf:.3f}   (should be ~0.5 — if much higher, LEAK)')

## Cell 10 — Subgroup tagging for error analysis

Classify each sockpuppet by how their IP-group peers interact in threads:
- **within_thread**: ≥1 peer co-posted in the same thread (what Karin's intra-thread TAG can already see)
- **cross_thread_only**: peers exist but never co-occur in a thread (your contribution should help here)
- **isolated**: solo group, no peers

Used in Phase III evaluation to slice F1 by subgroup. **This is the key plot of Chapter 6.**

In [ ]:
sock_group_members = {f'sock_{gid}': set(members) for gid, members in enumerate(sock_groups_master)}
user_threads = {uid: {t for t, _ in acts} for uid, acts in user_activity.items()}

rows = []
for uid, gid in user_to_group.items():
    if not gid.startswith('sock_'):
        continue
    peers = sock_group_members[gid] - {uid}
    my_threads = user_threads.get(uid, set())
    if not peers:
        sub = 'isolated'
    elif any(my_threads & user_threads.get(p, set()) for p in peers):
        sub = 'within_thread'
    else:
        sub = 'cross_thread_only'
    rows.append({'user_id': uid, 'ip_group': gid, 'subgroup': sub,
                 'group_size': len(sock_group_members[gid])})

sub_df = pd.DataFrame(rows)
sub_df.to_csv(OUT_DIR / 'sockpuppet_subgroups.csv', index=False)
print(sub_df.subgroup.value_counts())
print()
for exp in (1, 2, 3):
    test_sock = splits[exp]['test_sock']
    seg = sub_df[sub_df.user_id.isin(test_sock)].subgroup.value_counts().to_dict()
    print(f'Exp {exp} test sockpuppets ({len(test_sock)}): {seg}')

## Cell 11 — Sensitivity sweep (optional, for Ch 5 robustness table)

Wrap cells 5–8 in a function and call with a grid of `(MIN_SHARED_THREADS, TIME_WINDOWS_MIN)`:

```python
grid = [
    dict(min_shared=2, windows=[30, 120, 360]),  # default
    dict(min_shared=3, windows=[30, 120, 360]),
    dict(min_shared=5, windows=[30, 120, 360]),
    dict(min_shared=2, windows=[15, 60, 240]),
    dict(min_shared=2, windows=[60, 360, 1440]),
]
```

Aggregate Phase III F1 scores per config to show robustness.

## Next steps

1. **Smoke test on the sample**: set `POSTS_FILE = .../00_Datasets-sample.txt` and run end-to-end.
2. **Switch to the full `00_Datasets.txt`** and re-run.
3. **Cell 9 must pass.** Shuffled-label AUC near 0.5. If not, stop and debug.
4. **Join the output CSVs with your Karin + Lindsey user-level matrices** on `user_id`. Use a LEFT join from your existing feature table — coordination features only cover users with ≥1 qualifying co-occurrence; the rest get NaN, which you fill with 0 (or the column median, but 0 is the right semantic for 'no coordination signal').
5. **Run Phase III** with three feature sets per experiment:
   - Karin only
   - Karin + Lindsey
   - Karin + Lindsey + Coordination
6. **Use `sockpuppet_subgroups.csv` to slice F1** on the test set by `within_thread` vs `cross_thread_only`. Expected finding: coordination features help most on `cross_thread_only`.